# Phase 2 — Data
**Goal:** select the datasets with a fixed, written rule and turn them into clean, identical, ready-to-use files, so that every later experiment trains and tests on exactly the same data.

**What this notebook produces**
- `data/<name>.npz`: one file per dataset holding `X_train, y_train, X_test, y_test`
- `results/tables/dataset_selection.csv`: every candidate dataset and why it was kept or excluded
- `results/tables/datasets.csv`: the dataset table for the paper's *Methodology* section

**The big idea:** in Phase 5 we will add label noise **only to the training labels** and measure error on a **clean test set**. So the test set must be split off now, once, and never touched again. If we split differently in every experiment, differences in results could come from the split instead of the noise.

Kernel: **Python (labelnoise)**.

## Step 1 — Load the settings from Phase 0
We reuse the same project folder and the same `SEED = 42` that Phase 0 saved in `config.json`.

**Why:** if every notebook reads the seed from one file, all phases stay consistent and the whole study can be repeated exactly by someone else.

In [7]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

NB_DIR = Path.cwd()
BASE = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
config = json.loads((BASE / "config.json").read_text())
SEED = config["SEED"]

DATA_DIR = BASE / "data"
TABLE_DIR = BASE / "results" / "tables"
openml.config.set_root_cache_directory(str(DATA_DIR / "openml_cache"))   # raw downloads are kept here

print("Project folder:", BASE)
print("SEED:", SEED)

Project folder: /home/gravtion/VS Code/College/ML Research Paper
SEED: 42


## Step 2 — Select the datasets with a fixed rule
We don't hand-pick datasets. Choosing them by hand could look like **cherry-picking** (choosing data that gives nice results). Instead we start from **OpenML-CC18**, a curated, widely used benchmark suite of 72 classification datasets (OpenML suite ID 99), and keep **every** dataset that passes all of these rules:

| Rule | Reason |
|---|---|
| Exactly 2 classes | One-sided noise and the 0-1 bias–variance formula are uniquely defined for binary tasks |
| ≤ 50,000 rows | Keeps 51 repeated trainings × 13 noise conditions feasible on one PC |
| ≤ 100 features | Excludes special very-high-dimensional cases (e.g. madelon, Bioresponse) |
| No missing values | So the choice of how to fill gaps cannot mix with the label-noise effect |
| Minority class ≥ 15 % of rows | With fewer positives, 30 % one-sided noise would leave almost no class-1 examples, and accuracy becomes misleading |

In the paper this is one sentence: *"We used all binary OpenML-CC18 datasets with at most 50,000 rows, at most 100 features, no missing values and a minority class of at least 15 %."* Anyone can repeat the selection exactly.

The code below applies the rules and records **why each excluded dataset was dropped**. That record is part of being transparent.

In [8]:
suite = openml.study.get_suite(99)                         # OpenML-CC18
cand = openml.datasets.list_datasets(data_id=list(suite.data), output_format="dataframe")
cand["features"] = cand["NumberOfFeatures"] - 1           # OpenML counts the label as a feature
cand["minority_share"] = (cand["MinorityClassSize"] / cand["NumberOfInstances"]).round(3)

def exclusion_reason(r):
    if r.NumberOfClasses != 2:          return "not binary"
    if r.NumberOfInstances > 50_000:    return "more than 50,000 rows"
    if r.features > 100:                return "more than 100 features"
    if r.NumberOfMissingValues > 0:     return "has missing values"
    if r.minority_share < 0.15:         return "minority class below 15%"
    return ""

cand["excluded_because"] = cand.apply(exclusion_reason, axis=1)
cand["selected"] = cand["excluded_because"] == ""
selection = cand[["did", "name", "NumberOfInstances", "features", "NumberOfClasses",
                  "NumberOfMissingValues", "minority_share", "selected", "excluded_because"]]
selection = selection.sort_values(["selected", "NumberOfInstances"], ascending=[False, True])
selection.to_csv(TABLE_DIR / "dataset_selection.csv", index=False)

# Short, file-friendly names for the selected datasets
SHORT = {"blood-transfusion-service-center": "blood-transfusion"}
chosen = selection[selection["selected"]]
DATASETS = {SHORT.get(n, n): int(d) for n, d in zip(chosen["name"], chosen["did"])}

print("CC18 datasets:", len(cand), "| binary:", int((cand.NumberOfClasses == 2).sum()), "| selected:", len(DATASETS))
print("\nWhy binary CC18 datasets were excluded:")
print(cand[(cand.NumberOfClasses == 2) & ~cand.selected]["excluded_because"].value_counts().to_string())
chosen[["did", "name", "NumberOfInstances", "features", "minority_share"]]

CC18 datasets: 72 | binary: 35 | selected: 15

Why binary CC18 datasets were excluded:
excluded_because
minority class below 15%    8
has missing values          7
more than 100 features      4
more than 50,000 rows       1


,did,name,NumberOfInstances,features,minority_share
1063,1063,kc2,522.0,21.0,0.205
1510,1510,wdbc,569.0,30.0,0.373
1480,1480,ilpd,583.0,10.0,0.286
1464,1464,blood-transfusion-service-center,748.0,4.0,0.238
37,37,diabetes,768.0,8.0,0.349
50,50,tic-tac-toe,958.0,9.0,0.347
31,31,credit-g,1000.0,20.0,0.300
1494,1494,qsar-biodeg,1055.0,41.0,0.337
1462,1462,banknote-authentication,1372.0,4.0,0.445
1067,1067,kc1,2109.0,21.0,0.155


## Step 3 — Download one dataset and look at it
Before processing everything automatically, we look at one dataset by hand. **Always look at raw data first.** It is how you notice text columns, missing values or strange labels.

OpenML gives us:
- `X`: the features (a table of measurements; one row = one example)
- `y`: the label (the class we want to predict)
- `is_categorical`: which columns hold categories (text) instead of numbers

In [9]:
d = openml.datasets.get_dataset(DATASETS["credit-g"], download_data=True,
                                download_qualities=False, download_features_meta_data=False)
X_raw, y_raw, is_categorical, columns = d.get_data(target=d.default_target_attribute)

print("Rows:", X_raw.shape[0], "| Features:", X_raw.shape[1])
print("Categorical columns:", [c for c, cat in zip(columns, is_categorical) if cat])
print("Missing values:", int(X_raw.isna().sum().sum()))
print("Labels:", y_raw.value_counts().to_dict())
X_raw.head()

Rows: 1000 | Features: 20
Categorical columns: ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']
Missing values: 0
Labels: {'good': 700, 'bad': 300}


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,<0,6,critical/other existing credit,radio/tv,1169.0,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes
1,0<=X<200,48,existing paid,radio/tv,5951.0,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes
2,no checking,12,critical/other existing credit,education,2096.0,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes
3,<0,42,existing paid,furniture/equipment,7882.0,<100,4<=X<7,2,male single,guarantor,4,life insurance,45,none,for free,1,skilled,2,none,yes
4,<0,24,delayed previously,new car,4870.0,<100,1<=X<4,3,male single,none,4,no known property,53,none,for free,2,skilled,2,none,yes


**What to notice in credit-g:**
- Columns such as `checking_status` or `purpose` contain **words** (e.g. `'<0'`, `'radio/tv'`). Models need numbers, so these must be converted (Step 4).
- Labels are the words `good` / `bad`, not 0 / 1. We must convert them too.
- There are 700 good and 300 bad, so the classes are **imbalanced**.

## Step 4 — The preparation rules (applied to every dataset)
Each rule has a reason that matters for our experiment:

**Rule 1: Labels become 0 / 1, with the *minority* (smaller) class = 1.**
Every dataset then follows the same convention, so "one-sided noise flips class 1 → 0" always means *the rarer class gets missed*. That is the realistic case (e.g. sick patients recorded as healthy).

**Rule 2: Split 70 % train / 30 % test, *stratified*, with the fixed seed.**
*Stratified* means both parts keep the same class proportions as the full dataset. Without it, a small dataset could, by chance, get a test set with very few class-1 examples.

**Rule 3: Split *before* any processing, and learn the processing from the training part only.**
The imputer (fills missing values with the column median) and the one-hot encoder (turns text categories into 0/1 columns) are *fitted on the training data*, then applied to the test data. If we learned anything from the test set, we would be "peeking" at it, which is called **data leakage**, and our test results would be too optimistic.

**Rule 4: No scaling here.**
Tree-based models don't care about the scale of a feature (a split at "age > 40" works the same in years or months). Only the logistic-regression *reference model* used for boundary noise needs scaled features, and Phase 3 will scale inside that model.

**One-hot encoding example:** a column `purpose` with values `car`, `radio/tv`, `education` becomes three columns `purpose_car`, `purpose_radio/tv`, `purpose_education`, each 0 or 1.

In [10]:
def prepare(name, did, seed=SEED):
    d = openml.datasets.get_dataset(did, download_data=True,
                                    download_qualities=False, download_features_meta_data=False)
    X, y, is_cat, cols = d.get_data(target=d.default_target_attribute)

    # Rule 1: labels -> 0/1, minority class = 1
    counts = y.value_counts()
    minority = counts.idxmin()
    y01 = (y == minority).astype(int).to_numpy()

    # Rule 2: stratified 70/30 split with the fixed seed (done BEFORE any processing)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y01, test_size=0.30, stratify=y01, random_state=seed)

    # Rule 3: impute + one-hot encode, learned from the training part only
    cat_cols = [c for c, cat in zip(cols, is_cat) if cat]
    num_cols = [c for c, cat in zip(cols, is_cat) if not cat]
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ])
    X_tr = prep.fit_transform(X_tr).astype(np.float64)   # fit on train ...
    X_te = prep.transform(X_te).astype(np.float64)       # ... only apply to test
    feature_names = np.array(prep.get_feature_names_out(), dtype=str)

    info = {
        "dataset": name, "openml_id": did,
        "rows": len(y01), "raw_features": X.shape[1], "features_after_encoding": X_tr.shape[1],
        "categorical_features": len(cat_cols), "missing_values": int(X.isna().sum().sum()),
        "positive_class (label 1)": str(minority),
        "positive_share": round(y01.mean(), 3),
        "train_rows": len(y_tr), "test_rows": len(y_te),
    }
    return X_tr, y_tr, X_te, y_te, feature_names, info

## Step 5 — Prepare all selected datasets and save them
For each dataset we apply the rules and save one compressed `.npz` file (NumPy's format for several arrays in one file). Every later notebook loads these files, so **all experiments see identical data**.

The two largest datasets (PhishingWebsites, electricity) take a little longer to download the first time.

In [11]:
rows = []
for name, did in DATASETS.items():
    X_tr, y_tr, X_te, y_te, feats, info = prepare(name, did)
    np.savez_compressed(DATA_DIR / f"{name}.npz",
                        X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te, feature_names=feats)
    rows.append(info)
    print(f"saved {name:24s} train {X_tr.shape}  test {X_te.shape}")

summary = pd.DataFrame(rows)

saved kc2                      train (365, 21)  test (157, 21)
saved wdbc                     train (398, 30)  test (171, 30)
saved ilpd                     train (408, 11)  test (175, 11)
saved blood-transfusion        train (523, 4)  test (225, 4)
saved diabetes                 train (537, 8)  test (231, 8)
saved tic-tac-toe              train (670, 27)  test (288, 27)
saved credit-g                 train (700, 61)  test (300, 61)
saved qsar-biodeg              train (738, 41)  test (317, 41)
saved banknote-authentication  train (960, 4)  test (412, 4)
saved kc1                      train (1476, 21)  test (633, 21)
saved kr-vs-kp                 train (2237, 73)  test (959, 73)
saved spambase                 train (3220, 57)  test (1381, 57)
saved phoneme                  train (3782, 5)  test (1622, 5)
saved PhishingWebsites         train (7738, 68)  test (3317, 68)
saved electricity              train (31718, 14)  test (13594, 14)


## Step 6 — Sanity checks
A bug in data preparation silently ruins every later result, so we test the files we just saved, not the variables in memory.

| Check | Why it matters |
|---|---|
| No missing values left | Models crash or behave strangely on NaN |
| Labels are only 0 and 1 | The noise functions and the decomposition assume 0/1 |
| Class 1 is the minority | Rule 1 holds, so one-sided noise means the same thing everywhere |
| Train and test have the same class share (±2 %) | Stratification worked |
| Test rows ≈ 30 % | The split is correct |
| Loading twice gives identical arrays | Files are stable and the same for every phase |

In [12]:
problems = []
for name in DATASETS:
    f = np.load(DATA_DIR / f"{name}.npz", allow_pickle=False)
    Xtr, ytr, Xte, yte = f["X_train"], f["y_train"], f["X_test"], f["y_test"]
    n = len(ytr) + len(yte)
    checks = {
        "no NaN": not (np.isnan(Xtr).any() or np.isnan(Xte).any()),
        "labels 0/1": set(np.unique(np.r_[ytr, yte])) == {0, 1},
        "class 1 is minority": np.r_[ytr, yte].mean() < 0.5,
        "same class share": abs(ytr.mean() - yte.mean()) < 0.02,
        "test about 30%": abs(len(yte) / n - 0.30) < 0.01,
        "same columns": Xtr.shape[1] == Xte.shape[1],
        "reload identical": np.array_equal(Xtr, np.load(DATA_DIR / f"{name}.npz")["X_train"]),
    }
    failed = [k for k, ok in checks.items() if not ok]
    print(f"{name:24s} {'OK' if not failed else 'FAILED: ' + ', '.join(failed)}")
    problems += failed
print("\nAll checks passed." if not problems else "\nFix the failures above before Phase 3.")

kc2                      OK
wdbc                     OK
ilpd                     OK
blood-transfusion        OK
diabetes                 OK
tic-tac-toe              OK
credit-g                 OK
qsar-biodeg              OK
banknote-authentication  OK
kc1                      OK
kr-vs-kp                 OK
spambase                 OK
phoneme                  OK
PhishingWebsites         OK
electricity              OK

All checks passed.


## Step 7 — The dataset table for the paper
This table goes straight into the *Methodology* section (Phase 8). Reviewers use it to judge how general your results are.

In [13]:
summary.to_csv(TABLE_DIR / "datasets.csv", index=False)
print("Saved:", TABLE_DIR / "datasets.csv")
summary[["dataset", "openml_id", "rows", "raw_features", "features_after_encoding",
         "categorical_features", "positive_class (label 1)", "positive_share", "train_rows", "test_rows"]]

Saved: /home/gravtion/VS Code/College/ML Research Paper/results/tables/datasets.csv


,dataset,openml_id,rows,raw_features,features_after_encoding,categorical_features,positive_class (label 1),positive_share,train_rows,test_rows
0,kc2,1063,522,21,21,0,yes,0.205,365,157
1,wdbc,1510,569,30,30,0,2,0.373,398,171
2,ilpd,1480,583,10,11,1,2,0.286,408,175
3,blood-transfusion,1464,748,4,4,0,2,0.238,523,225
4,diabetes,37,768,8,8,0,tested_positive,0.349,537,231
5,tic-tac-toe,50,958,9,27,9,negative,0.347,670,288
6,credit-g,31,1000,20,61,13,bad,0.300,700,300
7,qsar-biodeg,1494,1055,41,41,0,2,0.337,738,317
8,banknote-authentication,1462,1372,4,4,0,2,0.445,960,412
9,kc1,1067,2109,21,21,0,True,0.155,1476,633


## How to read this table
- **Size range:** from about 500 rows (kc2) to about 45,000 (electricity). This spread lets Phase 7 also check whether **noise hurts small datasets more than large ones**. Theory says variance shrinks as data grows.
- **positive_share:** from about 0.15 (kc1) to 0.48 (kr-vs-kp), so we cover clearly imbalanced and nearly balanced data. This matters because **one-sided noise changes the class balance** (Phase 5 will report it).
- **features_after_encoding > raw_features:** six datasets have text categories that one-hot encoding expanded: ilpd (10 → 11), electricity (8 → 14), tic-tac-toe (9 → 27), credit-g (20 → 61), PhishingWebsites (30 → 68) and kr-vs-kp (36 → 73).
- **Small test sets** (kc2, wdbc, ilpd) give noisier results. That is one reason we average over all datasets and use statistical tests in Phase 7.

## Phase 2 checklist
- [ ] `results/tables/dataset_selection.csv` saved: every CC18 dataset with its keep/exclude reason
- [ ] 15 `.npz` files in `data/`
- [ ] All sanity checks print **OK**
- [ ] `results/tables/datasets.csv` saved and looked at

**Next: Phase 3**: models, noise injection and the bias–variance decomposition code.